# Bundle Adjustment — API Demo

How to invoke bundle adjustment (BA) on a small image set.

**Not an evaluation.** For benchmark results across `baseline / ba / lc`, see `eval_7scenes_gt.ipynb`. Compute lives in `evals/eval_gt.py`.

Two API styles are demonstrated:

- **Section A**: high-level wrapper — `BundleAdjustment(VGGTXCreator())`
- **Section B**: low-level manual — `extract_tracks_vggsfm` + `run_bundle_adjustment`

**Runtime target**: <90 s on a 5-frame sample. The full BA path (track extraction, LM iterations) is exercised, but the data is small enough to fit easily under any container memory cap.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import shutil
import tempfile
from pathlib import Path

import torch

from collab_splats.pointcloud import BundleAdjustment, BundleAdjustmentConfig
from collab_splats.pointcloud.bundle_adjustment import (
    extract_tracks_vggsfm,
    run_bundle_adjustment,
)
from collab_splats.pointcloud.feedforward import VGGTXCreator, _raw_to_world_points

/opt/conda/envs/nerfstudio/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/opt/conda/envs/nerfstudio/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/opt/conda/envs/nerfstudio/lib/python3.11/site-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_5m_224 in registry with mobile_sam.modeling.tiny_vit_sam.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/opt/conda/envs/nerfstudio/lib/python3.11/site-packages/mobile_sam/mod

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## §0 — Setup

Symlink 5 frames from 7-Scenes chess seq-01 into a temp dir. Source data is downloaded by `evals/download_7scenes.sh` (see `evals/`). No new committed binaries.

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATASET   = "birds_c0043"
METHOD    = "vggtx"    # "vggtx" | "mapanything"
VARIANT   = "ba"       # "ba" | "lc" | "" (empty = raw baseline)

CACHE  = Path("../../.cache") / DATASET
IMAGES = CACHE / "images"
RECON  = CACHE / METHOD / VARIANT if VARIANT else CACHE / METHOD
RECON.mkdir(parents=True, exist_ok=True)

In [4]:
CANNED_SOURCE = Path("/workspace/collab-splats/data/7scenes/chess/chess/seq-01")
N_FRAMES = 5

# Validate canned data exists before proceeding
assert CANNED_SOURCE.exists(), (
    f"Canned source not found at {CANNED_SOURCE}. "
    "Run `bash evals/download_7scenes.sh chess data/7scenes` and unzip the inner seq-01.zip first."
)

# Symlink N_FRAMES frames into a temp dir for the demo
DEMO_DIR = Path(tempfile.mkdtemp(prefix="ba_demo_"))
for i, src in enumerate(sorted(CANNED_SOURCE.glob("*.color.png"))[:N_FRAMES]):
    (DEMO_DIR / f"{i:06d}.png").symlink_to(src.resolve())

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="ba_demo_out_"))
print(f"demo image dir: {DEMO_DIR}  ({len(list(DEMO_DIR.iterdir()))} frames)")
print(f"output dir:     {OUTPUT_DIR}")

demo image dir: /tmp/ba_demo_3s0sfv23  (5 frames)
output dir:     /tmp/ba_demo_out_hakdvksq


## §1 — Section A: High-Level Wrapper API

`BundleAdjustment` wraps any feedforward creator. `reconstruct()` runs inference end-to-end, then refines poses and 3D points via Levenberg–Marquardt on VGGSfM track observations. This is the path used by `evals/eval_gt.py --conditions ba`.

In [5]:
# Wrap VGGTXCreator with BundleAdjustment and run end-to-end reconstruction
creator = BundleAdjustment(VGGTXCreator(), config=BundleAdjustmentConfig())
# equivalent factory: collab_splats.pointcloud.make_creator("vggtx", use_ba=True)

result = creator.reconstruct(DEMO_DIR, OUTPUT_DIR / "wrapper")
print(f"after BA: {len(result.points):,} points, {result.camera_poses.shape[0]} cameras")

[07:53:51] Loading model (cuda)...                                                                      ]8;id=1171451;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171452;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#541\541]8;;\

[07:55:05]   done in 74.7s                                                                              ]8;id=1171458;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171459;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#543\543]8;;\

           Preprocessing images...                                                                      ]8;id=1171465;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171466;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#547\547]8;;\

[07:55:06]   → 5 images  done in 0.4s                                                                   ]8;id=1171472;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171473;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#549\549]8;;\

           Running inference...                                                                         ]8;id=1171479;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171480;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#553\553]8;;\

...

[07:56:12] Building COLMAP reconstruction...                                                            ]8;id=1171507;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171508;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#569\569]8;;\

Warning: More than one camera is found in /tmp/ba_demo_out_hakdvksq/wrapper/colmap/sparse/0

{1: Camera(id=1, model='SIMPLE_PINHOLE', width=640, height=480, params=array([808.2104148 , 319.99999011, 240.00000238])), 2: Camera(id=2, model='SIMPLE_PINHOLE', width=640, height=480, params=array([798.57773834, 319.99999011, 240.00000238])), 3: Camera(id=3, model='SIMPLE_PINHOLE', width=640, height=480, params=array([793.64130932, 319.99999011, 240.00000238])), 4: Camera(id=4, model='SIMPLE_PINHOLE', width=640, height=480, params=array([798.91316304, 319.99999011, 240.00000238])), 5: Camera(id=5, model='SIMPLE_PINHOLE', width=640, height=480, params=array([795.20999291, 319.99999011, 240.00000238]))}


[07:56:35]   done in 22.9s                                                                              ]8;id=1171514;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171515;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#588\588]8;;\

after BA: 500,000 points, 5 cameras


In [6]:
# Save BA result to standard cache path
# Note: BundleAdjustment.reconstruct() returns PointcloudResult, not FeedforwardResult.
# PointcloudResult does not have save_zarr(); saving the colmap output_dir is the
# standard persistence mechanism for this wrapper. RECON holds the output path.
colmap_out = OUTPUT_DIR / "wrapper"
if colmap_out.exists():
    dest = RECON / "colmap"
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(colmap_out, dest, dirs_exist_ok=True)
    print(f"BA colmap result saved → {dest}")

BA colmap result saved → ../../.cache/birds_c0043/vggtx/ba/colmap


## §2 — Section B: Manual API

Same flow as Section A, exposed step-by-step:

1. Run inference and capture raw outputs
2. Extract VGGSfM tracks (cross-frame 2D correspondences + initial 3D points)
3. Run `run_bundle_adjustment` to refine poses + 3D points

Useful when you want to inspect or modify intermediate state.

In [7]:
# Run inference step-by-step using the raw VGGTXCreator API
raw_creator = VGGTXCreator()
raw_creator.load_model()
raw_creator.setup_inference(DEMO_DIR)
raw_creator.run_inference()

# Extract raw outputs: extrinsics, intrinsics, depth map, confidence
raw = raw_creator.raw_outputs
extrinsic = raw["extrinsic"]                  # (N, 3, 4) world-to-cam
intrinsic = raw["intrinsics_downsampled"]     # (N, 3, 3)
N = extrinsic.shape[0]
model_h, model_w = int(raw["depth"].shape[1]), int(raw["depth"].shape[2])

# Reconstruct world-space 3D point grid from depth + pose
wp_flat, _ = _raw_to_world_points(raw, subsample=1)
world_pts = wp_flat.reshape(N, model_h, model_w, 3)
conf = torch.from_numpy(raw["depth_conf"])

# Extract VGGSfM keypoint tracks: cross-frame 2D correspondences + 3D anchor points
tracks, vis_scores, pts3d_kp = extract_tracks_vggsfm(raw["images"], conf, world_pts)
print(f"tracks {tracks.shape}  vis_scores {vis_scores.shape}  pts3d_kp {pts3d_kp.shape}")

# Run Levenberg-Marquardt bundle adjustment on the extracted tracks
vis_mask = vis_scores > 0.5
pts3d_ref, ext_ref, intr_ref = run_bundle_adjustment(
    pts3d_kp, extrinsic, intrinsic, tracks, vis_mask,
    image_size=(model_h, model_w),
)
print(f"refined extrinsics {ext_ref.shape}  intrinsics {intr_ref.shape}")

[07:56:38] Loading model (cuda)...                                                                      ]8;id=1171520;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171521;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#541\541]8;;\

[07:57:52]   done in 73.9s                                                                              ]8;id=1171526;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171527;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#543\543]8;;\

           Preprocessing images...                                                                      ]8;id=1171532;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171533;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#547\547]8;;\

[07:57:53]   → 5 images  done in 0.4s                                                                   ]8;id=1171538;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171539;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#549\549]8;;\

           Running inference...                                                                         ]8;id=1171544;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=1171545;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#553\553]8;;\

...

Predicting tracks for query frame 1


Predicting tracks for query frame 4


Predicting tracks for query frame 2


Predicting tracks for query frame 3


tracks (5, 10240, 2)  vis_scores (5, 10240)  pts3d_kp (10240, 3)
refined extrinsics (5, 3, 4)  intrinsics (5, 3, 3)


## Where to go next

- **Benchmarks across baseline/BA/LC** — `evals/eval_gt.py` (compute) + `docs/pointcloud/eval_7scenes_gt.ipynb` (viz)
- **Loop closure path** — `collab_splats.pointcloud.LoopClosure`
- **Tunable knobs** — `BundleAdjustmentConfig` (`max_reproj_error`, `lm_steps`, `shared_camera`, `min_inliers_per_frame`)
- **Source** — `collab_splats/pointcloud/wrappers.py` (BundleAdjustment), `collab_splats/pointcloud/bundle_adjustment.py` (track extraction + LM solver)